# 03_load_towers.py

```text
WHAT : Builds one table of mobile-phone sites in the NT from three sources, so a later
       step can ask "is there any mobile site within X km of this community?".
READS: data/external/acma_rrl/site.csv, licence.csv, client.csv, device_details.csv
       data/raw/nt_mobile_coverage/mobile-coverage-all-sites.xlsx
       data/external/mbsp/*.kml
WRITES: data/processed/03_mobile_sites.csv        (one row per site; source column says which layer)
        data/processed/03_mobile_sites.geojson
        data/processed/03_mobile_sites.columns.md
WHY  : No single list is complete. The NT coverage guide is "a GUIDE only" and lists ~188
       sites. The ACMA licence register is the closest thing to a full carrier tower
       inventory (~458 NT cellular sites). The Mobile Black Spot Program KMLs show where
       new sites were funded. We keep all three, tagged by source, and let step 06 decide.
RUN  : python src/03_load_towers.py        (about 30 s - device_details.csv is ~2.1M rows)

"Carrier cellular site" (ACMA): a NT site whose devices sit under a licence that is both
held by a mobile carrier (client.LICENCEE matches a name in settings.ACMA_CARRIER_PATTERNS)
and in a mobile band (licence.LICENCE_CATEGORY_NAME in settings.ACMA_CELLULAR_BANDS). This
excludes point-to-point microwave backhaul, land mobile, aeronautical and broadcast licences.
```

In [106]:
import geopandas as gpd
import pandas as pd
import glob
import xml.etree.ElementTree as ET
import os

In [107]:
KML_NS = {"k": "http://www.opengis.net/kml/2.2"}
# Substrings that mark a mobile network operator in client.LICENCEE.
ACMA_CARRIER_PATTERNS = ("TELSTRA", "OPTUS", "VODAFONE", "TPG", "HUTCHISON", "PIVOTEL", "DENSE AIR")
# licence.LICENCE_CATEGORY_NAME values that are mobile-phone spectrum (not point-to-point
# microwave backhaul, land mobile, aeronautical, broadcast, or earth stations).
ACMA_CELLULAR_BANDS = {
    "700 MHz Band", "800 MHz Band", "850/900 MHz Band", "900 MHz Band", "1800 MHz Band",
    "1900 MHz Band", "2 GHz Band", "2.1 GHz Band", "2.3 GHz Band", "2.5 GHz Band",
    "3.4 GHz Band", "3.6 GHz Band", "26 GHz Band", "28 GHz Band",
    "PMTS Class B", "AWL - Standard",
}

def _carrier_of(licensee: str) -> str | None:
    """Return the carrier name if this licensee string is a mobile network operator."""
    text = (licensee or "").upper()
    for pattern in ACMA_CARRIER_PATTERNS:
        if pattern in text:
            return pattern.title()          # "TELSTRA" -> "Telstra"
    return None

In [108]:
site = pd.read_csv('data/external/acma_rrl/site.csv', dtype=str)
licence = pd.read_csv("data/external/acma_rrl/licence.csv", dtype=str,
                      usecols=["LICENCE_NO", "CLIENT_NO", "LICENCE_CATEGORY_NAME"])
client = pd.read_csv("data/external/acma_rrl/client.csv", dtype=str, usecols=["CLIENT_NO", "LICENCEE"])
devices = pd.read_csv("data/external/acma_rrl/device_details.csv", dtype=str, usecols=["LICENCE_NO", "SITE_ID"])
raw = pd.read_excel('data/raw/nt_mobile_coverage/mobile-coverage-all-sites.xlsx', header=None)

In [109]:
#NT sites with a usable coordinate.
site = site[site["STATE"] == "NT"].copy()
site["latitude"] = pd.to_numeric(site["LATITUDE"], errors="coerce")
site["longitude"] = pd.to_numeric(site["LONGITUDE"], errors="coerce")
site = site.dropna(subset=["latitude", "longitude"])
print(f"ACMA: {len(site)} NT sites in site.csv")

ACMA: 3771 NT sites in site.csv


In [110]:
# Licences that are carrier-held AND in a mobile band.
licence = licence.merge(client, on="CLIENT_NO", how="left")
licence["carrier"] = licence["LICENCEE"].map(_carrier_of)
mobile_licences = licence[
    licence["carrier"].notna()
    & licence["LICENCE_CATEGORY_NAME"].isin(ACMA_CELLULAR_BANDS)
]
print(f"ACMA: {len(mobile_licences)} carrier x mobile-band licences")

ACMA: 361 carrier x mobile-band licences


Devices link a licence to a site. Keep NT sites that host at least one such device, and record which carriers are present at each.

In [111]:
devices = devices[devices["SITE_ID"].isin(site["SITE_ID"])]
devices = devices.merge(mobile_licences[["LICENCE_NO", "carrier"]], on="LICENCE_NO", how="inner")


In [112]:
carriers_at_site = devices.groupby("SITE_ID")["carrier"].agg(lambda s: sorted(set(s)))
acma_sites = site[site["SITE_ID"].isin(carriers_at_site.index)].copy()
acma_sites["carrier_list"] = acma_sites["SITE_ID"].map(carriers_at_site)

In [113]:
acma_rows = pd.DataFrame({
    "source": "acma",
    "name": acma_sites["NAME"].fillna("").str.strip(),
    "latitude": acma_sites["latitude"].values,
    "longitude": acma_sites["longitude"].values,
    "site_type": "",
    "carriers": acma_sites["carrier_list"].apply(lambda lst: ";".join(lst)).values,
    "n_carriers": acma_sites["carrier_list"].apply(len).values,
    "guide_macro": False, "guide_small": False, "guide_proximity": False,
    "mbsp_round": "", "mbsp_is_small": False, "mbsp_built": False,
})
_by_carrier = pd.Series([c for lst in acma_sites["carrier_list"] for c in lst]).value_counts()
print(f"ACMA: {len(acma_rows)} NT carrier cellular sites  "
      f"({', '.join(f'{k} {int(v)}' for k, v in _by_carrier.items())})")

ACMA: 458 NT carrier cellular sites  (Telstra 325, Optus 162, Tpg 75, Vodafone 57, Pivotel 2)


### 2. NT mobile coverage guide spreadsheet.

   The real header is a few rows down (row 0 is a paragraph of explanation).

In [114]:
header_row = raw.index[raw.iloc[:, 0].astype(str).str.strip() == "SITE NAME"][0]
guide = raw.iloc[header_row + 1:].copy()
guide.columns = [str(c).strip() for c in raw.iloc[header_row]]
guide = guide.dropna(how="all")

In [115]:
for col in ("MACRO CELL", "SMALL CELL", "PROXIMITY TO CELL"):
    guide[col] = guide[col].astype(str).str.strip().str.upper().eq("YES")
guide["latitude"] = pd.to_numeric(guide["LATITUDE"], errors="coerce")
guide["longitude"] = pd.to_numeric(guide["LONGITUDE"], errors="coerce")
guide['population']= pd.to_numeric(guide["POPULATION"], errors="coerce").astype("Int64")
guide = guide.dropna(subset=["latitude", "longitude"])

In [116]:
guide_rows = pd.DataFrame({
    "source": "guide",
    "name": guide["SITE NAME"].astype(str).str.strip().values,
    "latitude": guide["latitude"].values,
    "longitude": guide["longitude"].values,
    "site_type": guide["SITE TYPE"].astype(str).str.strip().values,
    # PROVIDER is like "OPTUS/TELSTRA"; keep it for context (not used for tiering).
    "carriers": guide["PROVIDER"].fillna("").astype(str).str.replace("/", ";").str.title().values,
    "n_carriers": guide["PROVIDER"].fillna("").astype(str).apply(
        lambda s: len([p for p in s.split("/") if p.strip()])
    ).values,
    "population": guide["POPULATION"].values,
    "guide_macro": guide["MACRO CELL"].values,
    "guide_small": guide["SMALL CELL"].values,
    "guide_proximity": guide["PROXIMITY TO CELL"].values,
    "mbsp_round": "", "mbsp_is_small": False, "mbsp_built": False,
})
print(f"guide: {len(guide_rows)} sites  "
      f"(macro {guide_rows.guide_macro.sum()}, small {guide_rows.guide_small.sum()}, "
      f"proximity {guide_rows.guide_proximity.sum()})")

guide: 188 sites  (macro 62, small 35, proximity 98)


### 3. Mobile Black Spot Program funded base stations (KML, one file per round).

   Every site is listed twice inside its KML; keep the first per MBSP_ID.

In [117]:
MBSP_DIR = 'data/external/mbsp/'
seen: dict[str, dict] = {}
for path in sorted(glob.glob(os.path.join(MBSP_DIR, "*.kml"))):
    round_name = path.split(" - ")[1].replace(" Funded Base Stations.kml", "")
    for placemark in ET.parse(path).findall(".//k:Placemark", KML_NS):
        fields = {sd.get("name"): sd.text for sd in placemark.findall(".//k:SimpleData", KML_NS)}
        if fields.get("State") != "NT":
            continue
        coord_el = placemark.find(".//k:coordinates", KML_NS)
        if coord_el is None or not coord_el.text:
            continue
        mbsp_id = fields.get("MBSP_ID")
        if mbsp_id in seen:
            continue
        lon, lat = coord_el.text.strip().split()[0].split(",")[:2]
        station_type = (fields.get("Base_Station_Type") or fields.get("Solution_type") or "").strip()
        seen[mbsp_id] = {
            "source": "mbsp",
            "name": fields.get("Location") or fields.get("Site_Name") or mbsp_id,
            "latitude": float(lat),
            "longitude": float(lon),
            "site_type": station_type,
            "carriers": (fields.get("Grantee") or "").title(),
            "n_carriers": 1 if fields.get("Grantee") else 0,
            "guide_macro": False, "guide_small": False, "guide_proximity": False,
            "mbsp_round": round_name,
            "mbsp_is_small": "small" in station_type.lower() or "micro" in station_type.lower(),
            "mbsp_built": str(fields.get("Site_Status", "")).strip().lower() == "complete",
        }
mbsp_rows = pd.DataFrame(list(seen.values()))
print(f"MBSP: {len(mbsp_rows)} NT funded sites  "
      f"({int(mbsp_rows.mbsp_built.sum())} complete, {int(mbsp_rows.mbsp_is_small.sum())} small/micro)")

MBSP: 49 NT funded sites  (41 complete, 38 small/micro)


In [118]:
devices

,LICENCE_NO,SITE_ID,carrier
0,9469858,9003939,Optus
1,9469858,9003939,Optus
2,9469858,9003939,Optus
3,9469858,9003939,Optus
4,9469864,9003939,Optus
...,...,...,...
12419,10094267/248,135462,Optus
12420,10094267/248,9003939,Optus
12421,10094267/248,9003939,Optus
12422,13093516/1,37153,Telstra


In [119]:
site

,SITE_ID,LATITUDE,LONGITUDE,NAME,STATE,LICENSING_AREA_ID,POSTCODE,SITE_PRECISION,ELEVATION,HCIS_L2,latitude,longitude
0,1000,-12.471947,130.845073,Fort Hill Wharf DARWIN,NT,4,0800,Unknown,NaN,GO7H,-12.471947,130.845073
3,10000003,-12.464597,130.840708,6 Knuckey Street Darwin,NT,4,NaN,Within 10 metres,NaN,GO7H,-12.464597,130.840708
101,10000106,-12.387858,130.851172,33 Progress Dr Coconut Grove,NT,4,NaN,Within 10 metres,NaN,GO7H,-12.387858,130.851172
110,10000117,-12.538495,130.919425,INPEX Tower off Wickham Point Road BLAYDIN POINT,NT,4,NaN,Within 10 metres,NaN,GO7L,-12.538495,130.919425
196,10000204,-13.624853,131.456751,Douglas Station via 3377 Stuart Hwy DOUGLAS DALY,NT,5,NaN,Within 10 metres,NaN,GP2J,-13.624853,131.456751
...,...,...,...,...,...,...,...,...,...,...,...,...
127304,9910707,-19.770351,135.913591,Mobile Spectrum Licensing Site BARKLY HIGHWAY,NT,5,0862,Unknown,NaN,HR3P,-19.770351,135.913591
127310,9910713,-20.027881,137.511737,Mobile Spectrum Licensing Site BARKLY HIGHWAY,NT,5,0862,Unknown,NaN,IR5C,-20.027881,137.511737
127392,9910840,-23.150821,133.715405,Mobile Spectrum Licensing Site STUART HIGHWAY,NT,5,0872,Unknown,NaN,HS4C,-23.150821,133.715405
127393,9910841,-24.561618,133.240359,Mobile Spectrum Licensing Site STUART HIGHWAY,NT,5,0872,Unknown,NaN,HS7I,-24.561618,133.240359


In [120]:
mobile_licences

,LICENCE_NO,CLIENT_NO,LICENCE_CATEGORY_NAME,LICENCEE,carrier
4739,10066187/2,1136980,PMTS Class B,Vodafone Hutchison Australia Pty Limited,Vodafone
4990,10072096/7,1136980,PMTS Class B,Vodafone Hutchison Australia Pty Limited,Vodafone
5076,10072905/5,1136980,PMTS Class B,Vodafone Hutchison Australia Pty Limited,Vodafone
6108,10094267/248,512112,PMTS Class B,OPTUS MOBILE PTY LIMITED,Optus
6109,10094268/297,512112,PMTS Class B,OPTUS MOBILE PTY LIMITED,Optus
...,...,...,...,...,...
163390,9966101/3,1136980,PMTS Class B,Vodafone Hutchison Australia Pty Limited,Vodafone
163391,9966102/5,1136980,PMTS Class B,Vodafone Hutchison Australia Pty Limited,Vodafone
164104,9996195/9,1136980,PMTS Class B,Vodafone Hutchison Australia Pty Limited,Vodafone
164105,9996196/3,1136980,PMTS Class B,Vodafone Hutchison Australia Pty Limited,Vodafone


In [121]:
mbsp_rows

,source,name,latitude,longitude,site_type,carriers,n_carriers,guide_macro,guide_small,guide_proximity,mbsp_round,mbsp_is_small,mbsp_built
0,mbsp,Finke,-25.585000,134.573500,Macro cell,Telstra,1,False,False,False,Round 1,False,True
1,mbsp,Imanpa,-25.129090,132.569340,Macro cell,Telstra,1,False,False,False,Round 1,False,True
2,mbsp,Minjilang,-11.147870,132.573950,Macro cell,Telstra,1,False,False,False,Round 1,False,True
3,mbsp,Mt Liebig,-23.254500,131.277400,Macro cell,Telstra,1,False,False,False,Round 1,False,True
4,mbsp,Wallace Rockhole,-24.124480,133.087630,Macro cell,Telstra,1,False,False,False,Round 1,False,True
5,mbsp,Bark Hut Inn,-12.899765,131.675696,Small cell,Optus,1,False,False,False,Round 2,True,True
6,mbsp,Bynoe,-12.818780,130.620620,Macro cell,Telstra,1,False,False,False,Round 2,False,True
7,mbsp,Cape Crawford,-16.683400,135.726500,Small cell,Optus,1,False,False,False,Round 2,True,True
8,mbsp,Chinner Road,-12.985650,131.143540,Macro cell,Telstra,1,False,False,False,Round 2,False,True
9,mbsp,Devils Marbles Hotel,-20.640900,134.222700,Small cell,Optus,1,False,False,False,Round 2,True,True


In [122]:
guide_rows

,source,name,latitude,longitude,site_type,carriers,n_carriers,population,guide_macro,guide_small,guide_proximity,mbsp_round,mbsp_is_small,mbsp_built
0,guide,ADELAIDE RIVER,-13.23780,131.10473,VILLAGE,Optus;Telstra,2,317,True,False,False,,False,False
1,guide,AILERON STATION,-22.50000,133.24500,HIGHWAY,Optus;Telstra,2,NaN,False,False,True,,False,False
2,guide,ALAMIRRA,-11.06700,132.52800,COMMUNITY,Telstra,1,5,False,False,True,,False,False
3,guide,ALI CURUNG,-21.00333,134.40694,COMMUNITY,Telstra,1,394,True,False,False,,False,False
4,guide,ALPURRURULAM,-20.98417,137.84749,COMMUNITY,Telstra,1,350,True,False,False,,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,guide,YATHALAMARRA,-12.27400,134.92200,COMMUNITY,Telstra,1,59,False,False,True,,False,False
184,guide,YIRRKALA,-12.25279,136.89168,COMMUNITY,Telstra,1,524,True,False,False,,False,False
185,guide,YUDUYUDU,-12.23300,136.63700,COMMUNITY,Optus;Telstra,2,10,False,False,True,,False,False
186,guide,YUENDUMU,-22.25389,131.79529,COMMUNITY,Telstra,1,617,True,False,False,,False,False


In [123]:
guide

,SITE NAME,SITE TYPE,POPULATION,MACRO CELL,SMALL CELL,PROXIMITY TO CELL,PROVIDER,LATITUDE,LONGITUDE,latitude,longitude,population
5,ADELAIDE RIVER,VILLAGE,317,True,False,False,OPTUS/TELSTRA,-13.2378,131.10473,-13.23780,131.10473,317
6,AILERON STATION,HIGHWAY,NaN,False,False,True,OPTUS/TELSTRA,-22.5,133.245,-22.50000,133.24500,<NA>
7,ALAMIRRA,COMMUNITY,5,False,False,True,TELSTRA,-11.067,132.528,-11.06700,132.52800,5
8,ALI CURUNG,COMMUNITY,394,True,False,False,TELSTRA,-21.00333,134.40694,-21.00333,134.40694,394
9,ALPURRURULAM,COMMUNITY,350,True,False,False,TELSTRA,-20.98417,137.84749,-20.98417,137.84749,350
...,...,...,...,...,...,...,...,...,...,...,...,...
188,YATHALAMARRA,COMMUNITY,59,False,False,True,TELSTRA,-12.274,134.922,-12.27400,134.92200,59
189,YIRRKALA,COMMUNITY,524,True,False,False,TELSTRA,-12.25279,136.89168,-12.25279,136.89168,524
190,YUDUYUDU,COMMUNITY,10,False,False,True,OPTUS/TELSTRA,-12.233,136.637,-12.23300,136.63700,10
191,YUENDUMU,COMMUNITY,617,True,False,False,TELSTRA,-22.25389,131.79529,-22.25389,131.79529,617


In [124]:
acma_rows

,source,name,latitude,longitude,site_type,carriers,n_carriers,guide_macro,guide_small,guide_proximity,mbsp_round,mbsp_is_small,mbsp_built
759,acma,Telstra Site Anzac Oval Wills Terrace Alice Sp...,-23.696169,133.884366,,Telstra,1,False,False,False,,False,False
941,acma,Dead Bullock Soak Tanami,-20.533340,129.936940,,Telstra,1,False,False,False,,False,False
2321,acma,Defence Site Lattice Tower Off Tanami Road Bur...,-23.523950,133.680870,,Telstra,1,False,False,False,,False,False
2476,acma,Telstra Radio Site Lot 41 Yarralin,-16.448140,130.884212,,Telstra,1,False,False,False,,False,False
2610,acma,24 Grevillea Drive SADADEEN,-23.699840,133.894590,,Optus;Telstra;Tpg,3,False,False,False,,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
123115,acma,Telstra Temporary Fannie Bay Race Course FANNI...,-12.425501,130.844598,,Telstra,1,False,False,False,,False,False
123120,acma,Telstra Site - Temp. Bradshaw BRADSHAW,-15.573194,130.462080,,Telstra,1,False,False,False,,False,False
124446,acma,Telstra Site 26 Diarama Cl ARALUEN,-23.704516,133.853339,,Telstra,1,False,False,False,,False,False
125432,acma,Telstra Site Tanami Road Tanami TANAMI,-20.550268,130.333193,,Telstra,1,False,False,False,,False,False


## 4. Merge into one unified sites table

This is the merge the top-of-notebook docstring already promised (`WRITES:
03_mobile_sites.csv/.geojson/.columns.md`) but the notebook stopped short of doing.
`acma_rows`, `guide_rows`, and `mbsp_rows` share one schema (tagged by `source`), so this
just concatenates them and writes out CSV, GeoJSON, and a plain-English column reference
so nobody has to re-derive what each column means from the code later.


In [125]:
mobile_sites = pd.concat([acma_rows, guide_rows, mbsp_rows], ignore_index=True)

os.makedirs("data/new_processed", exist_ok=True)
mobile_sites.to_csv("data/new_processed/mobile_sites.csv", index=False)

gdf = gpd.GeoDataFrame(
    mobile_sites,
    geometry=gpd.points_from_xy(mobile_sites["longitude"], mobile_sites["latitude"]),
    crs="EPSG:4326",
)
gdf.to_file("data/new_processed/mobile_sites.geojson", driver="GeoJSON")

column_notes = {
    "source": "Which layer this row came from: acma (licensed carrier tower), guide "
              "(NT coverage guide entry), or mbsp (Mobile Black Spot Program funded site).",
    "name": "Site or place name as given by the source.",
    "latitude / longitude": "WGS84 coordinates.",
    "site_type": "Meaning differs by source -- for 'guide' rows this is the PLACE type "
                 "(VILLAGE/COMMUNITY/HIGHWAY/TOURISM); for 'mbsp' rows this is the "
                 "INFRASTRUCTURE type (Macro/Small/Micro cell); for 'acma' rows it is "
                 "blank (not recorded in the licence register). Don't group across "
                 "sources on this column without filtering by 'source' first.",
    "carriers": "Semicolon-separated carrier name(s) at the site (acma/guide), or the "
                "MBSP grantee (mbsp).",
    "n_carriers": "Count of carriers in the 'carriers' field.",
    "population": "Only populated for 'guide' rows (from the coverage guide's POPULATION "
                  "column); NaN elsewhere.",
    "guide_macro / guide_small / guide_proximity": "Coverage-guide flags; only meaningful "
                                                    "for 'guide' rows.",
    "mbsp_round": "MBSP funding round label; only populated for 'mbsp' rows.",
    "mbsp_is_small": "Whether the funded MBSP station is small/micro cell; only populated "
                     "for 'mbsp' rows.",
    "mbsp_built": "Whether the MBSP site's status was 'Complete'; only populated for "
                  "'mbsp' rows.",
}
with open("data/new_processed/03_mobile_sites.columns.md", "w") as f:
    f.write("# 03_mobile_sites.csv -- column reference\n\n")
    for col, note in column_notes.items():
        f.write(f"- **{col}**: {note}\n")

print(f"mobile_sites: {len(mobile_sites)} rows -> data/processed/03_mobile_sites.[csv|geojson]")
mobile_sites["source"].value_counts()


mobile_sites: 695 rows -> data/processed/03_mobile_sites.[csv|geojson]


source
acma     458
guide    188
mbsp      49
Name: count, dtype: int64

## 5. A second file, purpose-built for the challenge: community connectivity priority

`mobile_sites` above is a complete *inventory* -- good for mapping, good for step 06. But
the challenge question is "which communities have a gap and how urgent is it", and the
most direct answer joins:

- **guide** -- the only source with a named *community*, a population figure, and an
  existing macro/small/proximity coverage read
- **acma** -- the fullest tower inventory, used to compute how far each community actually
  sits from real carrier infrastructure
- **mbsp** -- whether that gap already has funding committed, and whether the funded site
  is built yet

Distance is straight-line (haversine), not drive time or a validated signal radius --
good enough for triage/prioritisation, not for claiming a community is or isn't "covered".


In [ ]:
import numpy as np


def haversine_km(lat1, lon1, lat2, lon2):
    """
    Distance in km between two points on the Earth's surface.

    The Earth is round, so we can't just use Pythagoras on latitude/longitude.
    This formula follows the curve of the planet instead.

    Inputs are in degrees. lat2/lon2 can be whole arrays (e.g. every tower),
    in which case we get back one distance per tower.

    It is a straight-line distance over the ground: no roads, no hills,
    no signal strength. Good for triage ("which tower is closest?"),
    not for claiming a place is or isn't covered.
    """
    R = 6371.0   # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])  # degrees -> radians for sin/cos
    dlat, dlon = lat2 - lat1, lon2 - lon1   # how far north/south and east/west
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))  # turn that into an angle, then angle x radius = km


acma_lat, acma_lon = acma_rows["latitude"].to_numpy(), acma_rows["longitude"].to_numpy()
mbsp_lat, mbsp_lon = mbsp_rows["latitude"].to_numpy(), mbsp_rows["longitude"].to_numpy()

nearest_tower_km, nearest_tower_carrier = [], []
nearest_mbsp_km, nearest_mbsp_round, nearest_mbsp_built = [], [], []

for lat, lon in zip(guide_rows["latitude"], guide_rows["longitude"]):
    d_acma = haversine_km(lat, lon, acma_lat, acma_lon)
    i = np.argmin(d_acma)
    nearest_tower_km.append(d_acma[i])
    nearest_tower_carrier.append(acma_rows["carriers"].iloc[i])

    d_mbsp = haversine_km(lat, lon, mbsp_lat, mbsp_lon)
    j = np.argmin(d_mbsp)
    nearest_mbsp_km.append(d_mbsp[j])
    nearest_mbsp_round.append(mbsp_rows["mbsp_round"].iloc[j])
    nearest_mbsp_built.append(mbsp_rows["mbsp_built"].iloc[j])

community_priority = pd.DataFrame({
    "name": guide_rows["name"],
    "latitude": guide_rows["latitude"],
    "longitude": guide_rows["longitude"],
    "site_type": guide_rows["site_type"],
    "population": guide_rows["population"],
    "has_macro": guide_rows["guide_macro"],
    "has_small": guide_rows["guide_small"],
    "proximity_only": guide_rows["guide_proximity"],
    "nearest_tower_km": np.round(nearest_tower_km, 1),
    "nearest_tower_carrier": nearest_tower_carrier,
    "nearest_mbsp_km": np.round(nearest_mbsp_km, 1),
    "nearest_mbsp_round": nearest_mbsp_round,
    "nearest_mbsp_built": nearest_mbsp_built,
})

# Starting-point triage flag, not a validated verdict: no macro/small coverage AND the
# nearest real tower is >15 km away. Tune 15 km against whatever the guide treats as
# "in range" before using this to rank communities in your submission.
community_priority["likely_gap"] = (
    ~community_priority["has_macro"]
    & ~community_priority["has_small"]
    & (community_priority["nearest_tower_km"] > 15)
)

community_priority = community_priority.sort_values(
    ["likely_gap", "population"], ascending=[False, False]
)
community_priority.to_csv("data/new_processed/community_connectivity_priority.csv", index=False)

print(f"community_priority: {len(community_priority)} communities, "
      f"{int(community_priority.likely_gap.sum())} flagged likely_gap")
community_priority.head(10)


community_priority: 188 communities, 10 flagged likely_gap


,name,latitude,longitude,site_type,population,has_macro,has_small,proximity_only,nearest_tower_km,nearest_tower_carrier,nearest_mbsp_km,nearest_mbsp_round,nearest_mbsp_built,likely_gap
119,NTURIYA,-22.12379,133.26529,COMMUNITY,66,False,False,True,15.7,Pivotel,101.9,Round 5,True,True
35,DHIPIRRINJURA,-12.24806,135.09757,COMMUNITY,30,False,False,True,20.6,Telstra,161.8,Round 7,False,True
161,UNDOOLYA BORE,-23.74800,134.11800,COMMUNITY,16,False,False,True,22.8,Optus;Tpg,37.9,Round 4,True,True
106,MUMUKALA,-12.61100,132.66600,COMMUNITY,12,False,False,True,19.4,Telstra,37.8,Round 5,True,True
90,MANDEDJKADJANG,-12.26454,134.03056,COMMUNITY,10,False,False,True,31.8,Telstra,117.8,Round 5,True,True
28,BUNHANURA,-12.37449,135.72221,COMMUNITY,7,False,False,True,16.9,Telstra,106.9,Round 7,False,True
174,WILGI,-11.49600,132.74900,COMMUNITY,6,False,False,True,43.2,Telstra,43.2,Round 1,True,True
80,KUY,-14.03546,129.59784,COMMUNITY,5,False,False,True,24.3,Telstra,151.8,Round 2,True,True
85,LIZARD BORE,-23.30034,131.13465,COMMUNITY,5,False,False,True,15.4,Telstra,15.4,Round 1,True,True
1,AILERON STATION,-22.50000,133.24500,HIGHWAY,NaN,False,False,True,18.9,Telstra,114.7,Round 2,True,True
